In [1]:
import numpy as np
import math
import random
import csv

In [ ]:
def P_action_softmax(value,gamma,state,action):
    Q = value[:,0]
    exp_Q = np.sum(np.exp(gamma*Q))
    return (np.exp(gamma*Q)/exp_Q)[action]

def P_action_con1(value,gamma):
    Q = value[:,1]
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q

def P_action_con2(value,gamma):
    Q = value[:,2]
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q
    
def P_action_con0(value,gamma):
    Q1 = (value[0,1]+value[0,2])/2
    Q2 = (value[1,1]+value[1,2])/2
    Q = np.array([Q1,Q2])
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q

def learn(value,lr,states,actions,reward):
    if states==[0,1,0] or states==[0,0,1]:
        value[1,0] = value[1,0] + lr * (reward - 1 - value[1,0])
    else :
        value[0,0] = value[0,0] + lr * (reward - value[0,0])
    if actions==1:
        if states==[0,1,0]:
            value[1,1] = value[1,1] + lr*(reward-value[1,1])      
        elif states==[0,0,1]:
            value[1,2] = value[1,2] + lr*(reward-value[1,2])    
        else:
            value[1,1] = value[1,1] + 0.5*lr*(reward-value[1,1])      
            value[1,2] = value[1,2] + 0.5*lr*(reward-value[1,2])
    return value

def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

def P_stay_cue_action_rl(value,gamma):
    P_stay = P_action_softmax(value,gamma,[1,0,0],0)
    P_cue = 1-P_stay
    action = np.random.choice([0,1],p=[P_stay,P_cue])
    return action

def P_safe_risk_action_rl(value,gamma,con):
    if con==0:
        state = [0,0.5,0.5]
        P_safe = P_action_con0(value,gamma)[0]
    elif con==1:
        state = [0,1,0]
        P_safe = P_action_con1(value,gamma)[0]
    elif con==2:
        state=[0,0,1]
        P_safe = P_action_con2(value,gamma)[0]
    else:
        print('ERROR')
    P_risk = 1-P_safe
    action = np.random.choice([0,1],p=[P_safe,P_risk])
    return action

def P_model_free_rl(gamma,lr):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

In [ ]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A

def a_update_m(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        #safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
        #safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def P_action_stay_cue_m_rl(A,gamma,action):
    preference = np.array([6,12,9,3,0,0,-1,-1])
    Q_risk_H = np.dot(A[:,2],preference)
    Q_risk_L = np.dot(A[:,3],preference)
    Q_stay = max(6,(Q_risk_H+Q_risk_L)/2)
    Q_cue = (max(5,Q_risk_H-1)+max(5,Q_risk_L-1))/2
    Q_stay_cue = np.array([Q_stay,Q_cue])
    exp_Q = np.sum(np.exp(gamma*Q_stay_cue))
    return (np.exp(gamma*Q_stay_cue)/exp_Q)[action]

def P_stay_cue_action_m_rl(a,gamma):
    A = dir(a)
    P_stay = P_action_stay_cue_m_rl(A,gamma,0)
    P_cue = 1-P_stay
    action = np.random.choice([0,1],p=[P_stay,P_cue])
    return action

def P_action_safe_risk_m_rl(A,gamma,result_stay_cue,action):
    preference = np.array([6,12,9,3,0,0,-1,-1])
    Q_risk_H = np.dot(A[:,2],preference)
    Q_risk_L = np.dot(A[:,3],preference)
    Q_safe = 6
    if result_stay_cue == 1:
        Q_safe_risk = np.array([Q_safe,Q_risk_H])
    elif result_stay_cue == 2:
        Q_safe_risk = np.array([Q_safe,Q_risk_L])
    else:
        Q_safe_risk = np.array([Q_safe,(Q_risk_H+Q_risk_L)/2])
    exp_Q = np.sum(np.exp(gamma*Q_safe_risk))    
    return (np.exp(gamma*Q_safe_risk)/exp_Q)[action]

def P_safe_risk_action_m_rl(a,gamma,con):
    A = dir(a)
    P_safe = P_action_safe_risk_m_rl(A,gamma,con,0)
    P_risk = 1-P_safe
    action = np.random.choice([0,1],p=[P_safe,P_risk])
    return action

def P_model_based_RL(prior,rate,gamma):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue_m_rl(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk_m_rl(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update_m(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

In [2]:
def Behavioral_Data(n):
    stay_cue = []
    safe_risk = []
    no_ask = []
    safe_risk_0 = []
    behavior_data = []
    can_ask = []
    fname = 'E:/practice/behavioral/uncertainty_' + str(n) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    no_ask.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    no_ask.append(1)
                else :
                    print('ERROR')
                stay_cue.append(int(line.split(',')[3]))
                safe_risk.append(int(float(line.split(',')[6])))
                safe_risk_0.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    can_ask.append(0)
                else :
                    can_ask.append(1)
    behavior_data.append(stay_cue)
    behavior_data.append(safe_risk)
    behavior_data.append(safe_risk_0)
    behavior_data.append(no_ask)
    behavior_data.append(can_ask)
    return behavior_data

In [4]:
discount = 0.1
preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A
def cum(a):
    a_0 = np.sum(a,axis=0)
    a_cum = np.array([np.ones([1,8])*a_0[0],
                      np.ones([1,8])*a_0[1],
                      np.ones([1,8])*a_0[2],
                      np.ones([1,8])*a_0[3],
                      np.ones([1,8])*a_0[4],
                      np.ones([1,8])*a_0[5],
                      np.ones([1,8])*a_0[6],
                      np.ones([1,8])*a_0[7]]).squeeze().T
    return a_cum
def H_entropy(A):
    H = np.matmul(A.T,np.log(A+np.e**(-16)))
    H = np.diag(H)
    return H
def G_ExperctedFreeEnergy(A,a,s,no_ask,p_al,p_ai,p_ex):
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    if no_ask == 0:
        return float(discount*(p_al*AL + p_ai*AI) - p_ex*EX)
    elif no_ask == 1:
        return float(p_al*AL + p_ai*AI - p_ex*EX)
    else :
        print('ERROR')
    
def P_stay_cue(A,a,pi,can_not,p_al,p_ai,p_ex):#stay,cue,0,stay-safe,1,stay-risk,2,cue-safe,3,cue-risk
    if can_not == 0:
        return 1
    else:
        G_pi = []
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0.5,0.5,0,0,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)))
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0,0,0.5,0.5,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)))
        s1 = np.array([0,0,0,0,0,0,0.5,0.5])
        s2 = np.array([1,0,0,0,0,0,0,0])
        s3 = np.array([0,0,0,0,0,0,0.5,0.5])
        s4 = np.array([0,1,0,0,0,0,0,0])
        G_pi.append(min((G_ExperctedFreeEnergy(A,a,s1,1,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,1,p_al,p_ai,p_ex)),(G_ExperctedFreeEnergy(A,a,s3,1,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s4,1,p_al,p_ai,p_ex))))     
        # G_pi.append((G_ExperctedFreeEnergy(A,a,s1,1,radio)+G_ExperctedFreeEnergy(A,a,s2,1,radio)+G_ExperctedFreeEnergy(A,a,s3,1,radio)+G_ExperctedFreeEnergy(A,a,s4,1,radio))/2)        
        s1 = np.array([0,0,0,0,0,0,0.5,0.5])
        s2 = np.array([0,0,1,0,0,0,0,0])
        s3 = np.array([0,0,0,0,0,0,0.5,0.5])
        s4 = np.array([0,0,0,1,0,0,0,0])
        G_pi.append(min((G_ExperctedFreeEnergy(A,a,s1,1,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,1,p_al,p_ai,p_ex)),(G_ExperctedFreeEnergy(A,a,s3,1,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s4,1,p_al,p_ai,p_ex))))
        # G_pi.append((G_ExperctedFreeEnergy(A,a,s1,1,radio)+G_ExperctedFreeEnergy(A,a,s2,1,radio)+G_ExperctedFreeEnergy(A,a,s3,1,radio)+G_ExperctedFreeEnergy(A,a,s4,1,radio))/2)
        if G_pi[0]<-20:
            G_pi[0]=-20
        if G_pi[1]<-20:
            G_pi[1]=-20
        if G_pi[2]<-20:
            G_pi[2]=-20
        if G_pi[3]<-20:
            G_pi[3]=-20
        exp_G = [np.exp(-G_pi[0]),np.exp(-G_pi[1]),np.exp(-G_pi[2]),np.exp(-G_pi[3])]
        total = exp_G[0]+exp_G[1]+exp_G[2]+exp_G[3]
        P = [(exp_G[0]+exp_G[1])/total,(exp_G[2]+exp_G[3])/total]
        return P[pi].squeeze()
def P_safe_risk(A,a,pi,con,p_al,p_ai,p_ex):#con=0,no,con=1,HRC,con=2,LRC,pi=0,safe,pi=1,risky
    G_pi = []
    if con !=0:
        noask = 1
    else :
        noask = 0
    if con == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
    elif con == 1:
        s=np.array([1,0,0,0,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
        s=np.array([0,0,1,0,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
    else :
        s=np.array([0,1,0,0,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
        s=np.array([0,0,0,1,0,0,0,0])
        G_pi.append((G_ExperctedFreeEnergy(A,a,s,noask,p_al,p_ai,p_ex)))
    if G_pi[0]<-20:
        G_pi[0]=-20
    if G_pi[1]<-20:
        G_pi[1]=-20
    exp_G = [np.exp(-G_pi[0]),np.exp(-G_pi[1])]
    total = exp_G[0]+exp_G[1]
    P_pi = [exp_G[0]/total,exp_G[1]/total]
    return P_pi[pi].squeeze()
def a_update(choice_1,choice_2,choice_3,rate,prior):#stay_cue,safe_risk,safe_risk_0
    a_0=np.array([[100.0,100,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,prior,prior,0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    aa=np.zeros((120,8,8))
    A = np.zeros((120,8,8))
    aa[0,:,:]=a_0
    A[0,:,:]=dir(a_0)
    a=a_0
    for i in range(0,119):
        if choice_1[i]==0 and choice_2==6 and choice_3[i]==0:
            s=np.array([0.5,0.5,0,0,0,0,0,0])
#safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
            o=np.array([1,0,0,0,0,0,0,0])
#safe,riskyHR,riskyLR,stay,cueHR,cueLR
        elif choice_1[i]==0 and choice_2[i]==0:
            s=np.array([0,0,0.5,0.5,0,0,0,0])
            o=np.array([0,0,0,0,1,0,0,0])
        elif choice_1[i]==0 and choice_2[i]==3:
            s=np.array([0,0,0.5,0.5,0,0,0,0])
            o=np.array([0,0,0,1,0,0,0,0])
        elif choice_1[i]==0 and choice_2[i]==6 and choice_3[i]==1:
            s=np.array([0,0,0.5,0.5,0,0,0,0])
            o=np.array([1,0,0,0,0,0,0,0])
        elif choice_1[i]==0 and choice_2[i]==9:
            s=np.array([0,0,0.5,0.5,0,0,0,0])
            o=np.array([0,0,1,0,0,0,0,0])
        elif choice_1[i]==0 and choice_2[i]==12:
            s=np.array([0,0,0.5,0.5,0,0,0,0])
            o=np.array([0,1,0,0,0,0,0,0])
        elif choice_1[i]==1 and choice_2[i]==6 and choice_3[i]==0:
            s=np.array([1,0,0,0,0,0,0,0])
            o=np.array([1,0,0,0,0,0,0,0])
        elif choice_1[i]==1 and choice_2[i]==0:
            s=np.array([0,0,1,0,0,0,0,0])
            o=np.array([0,0,0,0,1,0,0,0])
        elif choice_1[i]==1 and choice_2[i]==3:
            s=np.array([0,0,1,0,0,0,0,0])
            o=np.array([0,0,0,1,0,0,0,0])
        elif choice_1[i]==1 and choice_2[i]==6 and choice_3[i]==1:
            s=np.array([0,0,1,0,0,0,0,0])
            o=np.array([1,0,0,0,0,0,0,0])
        elif choice_1[i]==1 and choice_2[i]==9:
            s=np.array([0,0,1,0,0,0,0,0])
            o=np.array([0,0,1,0,0,0,0,0])      
        elif choice_1[i]==1 and choice_2[i]==12:
            s=np.array([0,0,1,0,0,0,0,0])
            o=np.array([0,1,0,0,0,0,0,0])
        elif choice_1[i]==2 and choice_2[i]==6 and choice_3[i]==0:
            s=np.array([0,1,0,0,0,0,0,0])
            o=np.array([1,0,0,0,0,0,0,0])
        elif choice_1[i]==2 and choice_2[i]==0:
            s=np.array([0,0,0,1,0,0,0,0])
            o=np.array([0,0,0,0,1,0,0,0])
        elif choice_1[i]==2 and choice_2[i]==3:
            s=np.array([0,0,0,1,0,0,0,0])
            o=np.array([0,0,0,1,0,0,0,0])
        elif choice_1[i]==2 and choice_2[i]==6 and choice_3[i]==1:
            s=np.array([0,0,0,1,0,0,0,0])
            o=np.array([1,0,0,0,0,0,0,0])
        elif choice_1[i]==2 and choice_2[i]==9:
            s=np.array([0,0,0,1,0,0,0,0])
            o=np.array([0,0,1,0,0,0,0,0])    
        else :
            s=np.array([0,0,0,1,0,0,0,0])
            o=np.array([0,1,0,0,0,0,0,0])
        if choice_1[i]!=0:
            a=a+rate*np.outer(o,s)#rate:learning rate
        else :
            a=a+rate*discount*np.outer(o,s)
        aa[i+1,:,:]=a
        A[i+1,:,:]=dir(a)
    return aa,A
def P_policy(Stay_cue,Safe_risk,Safe_risk_0,No_ask,Can_ask,rate,prior,p_al,p_ai,p_ex):
    a_policy, A_policy = a_update(Stay_cue,Safe_risk,Safe_risk_0,rate,prior)
    log_p_policy = 0
    for i in range(120):
        log_p_policy += np.log(P_stay_cue(A_policy[i],a_policy[i],No_ask[i],Can_ask[i],p_al,p_ai,p_ex)) + np.log(P_safe_risk(A_policy[i],a_policy[i],Safe_risk_0[i],Stay_cue[i],p_al,p_ai,p_ex))
        # print(P_stay_cue(A_policy[i],a_policy[i],No_ask[i],Can_ask[i]) * P_safe_risk(A_policy[i],a_policy[i],Safe_risk_0[i],Stay_cue[i]))
        # if i%10 == 0:
        #     p_policy = p_policy*100
    return log_p_policy.squeeze()

In [5]:
para_1 = [9.999995116397761, 0.2, 0, 0.15213857035611345, 4.603915737096504]
para_2 = [9.999938153297316, 0.2, 0.9649149918186556, 0.6363871535468617, 4.379745791995678]
para_3 = [2.655233325001383, 0.5691300680089594, 1.1771862119059617, 6.99938935338215, 3.615189420642945]
para_4 = [0.25, 19.999848760096423, 0, 0, 7.999983608076133]
para_5 = [0.25, 3.198255732332684, 7.999953660522862, 0, 7.544120924826154]
para_6 = [4.2872168318403086, 2.254268347716751, 3.5672001886595366, 2.0893928044837815, 7.999896205940316]
para_7 = [9.999881059334106, 0.2, 0.6120845156622792, 2.370447785581953, 3.650339994025964]
para_8 = [0.25001305743922797, 0.37183631839153214, 2.0155357799967626, 3.211229173333084, 7.068388281288445]
para_9 = [0.5791901118400372, 0.389606948188708, 0, 1.741853304386811, 7.037648630196096]
para_10 = [1.8472964464905945, 4.899998363877973, 0, 0.9621592391377676, 7.999999107952692]
para_11 = [9.999932142529534, 0.2, 0.22837820493758393, 5.7043575960956066, 5.088278674641351]
para_12 = [7.687421838618912, 0.25082355004850404, 4.363533923986321, 2.011200412014452, 5.027190347380737]
para_13 = [0.34307303505467235, 2.5213590487153414, 0, 0, 7.999990172396314]
para_14 = [0.45444294313607453, 0.2611309660778238, 0.8051129498588461, 0, 4.155240915175109]
para_15 = [4.56976931370617, 0.2, 7.999887128969976, 0.14361348313203204, 6.135737922404253]
para_16 = [0.8585051257741594, 0.5239600034053236, 1.1749556658097267, 1.6145879368629603, 7.270532709518294]
para_17 = [2.4813727134834194, 9.684269936439833, 1.789692853069645, 0, 7.999967083464736]
para_18 = [0.3454816658920618, 0.28143643668140145, 1.1176747600938535, 1.3795938661341796, 7.999970112212883]
para_19 = [5.66474652609509, 1.8933020420543816, 5.7401035695669025, 7.999988598990393, 7.9999362525874576]
para_20 = [0.25, 5.493331262271089, 7.99998444891321, 0, 7.999885813706375]
para_21 = [5.4309372583762645, 4.255564645120421, 5.489206375322779, 4.913216078133795, 5.612124302942743]
para_22 = [1.0530222794005286, 1.0820231888486145, 2.738689415092223, 3.1049054308888384, 7.999998727499435]
para_23 = [0.7214893258174525, 8.849680254155075, 7.99978713710951, 0, 7.999991879076482]
para_24 = [9.999974676179864, 0.2, 0.2535604126074203, 6.591032586247423, 2.996119152968295]
para_25 = [6.23797621811312, 1.8367636063414305, 4.6550920915826755, 0, 2.0505323070306343]

In [6]:
behavior_data = Behavioral_Data(1)
log_P0 = P_policy(behavior_data[0],behavior_data[1],behavior_data[2],behavior_data[3],behavior_data[4],para_1[0],para_1[1],para_1[2],para_1[3],para_1[4])
print(log_P0)

-109.09717817263602


In [7]:
for i in range(25):
    behavior_data = Behavioral_Data(i+1)
    log_P0 = P_policy(behavior_data[0],behavior_data[1],behavior_data[2],behavior_data[3],behavior_data[4],para_1[0],para_1[1],para_1[2],para_1[3],para_1[4])
    print("LL SUB("+str(i+1)+"):",log_P0)

LL SUB(1): -109.09717817263602
LL SUB(2): -104.61002276243224
LL SUB(3): -99.01413853725776
LL SUB(4): -214.15340721397038
LL SUB(5): -163.172262105864
LL SUB(6): -117.41191464318125
LL SUB(7): -96.75647588154511
LL SUB(8): -100.63682083252806
LL SUB(9): -105.72621134856459
LL SUB(10): -167.99009098847367
LL SUB(11): -84.38943406876491
LL SUB(12): -83.65108243872655
LL SUB(13): -152.32671450920168
LL SUB(14): -147.92605407125546
LL SUB(15): -100.40321804462104
LL SUB(16): -96.83788007452857
LL SUB(17): -160.548688335014
LL SUB(18): -64.42890529092
LL SUB(19): -50.84923947885084
LL SUB(20): -174.8328110376942
LL SUB(21): -83.82179818313155
LL SUB(22): -115.35150782593351
LL SUB(23): -166.02607373797983
LL SUB(24): -107.89355984164311
LL SUB(25): -161.93862782559614
